# 04 Data Integration

For this notebook, it will combining the cleaned datasets from Yahoo Finance, Wikipedia, and FRED into a shared project dataset. The goal is to prepare on clean set of files that can then be used for feature engiener, modeling, portfolio optimization, and dashboard work.

# Imports

In [1]:
import pandas as pd
import numpy as np 
from pathlib import Path

# Paths

In [2]:
YFINANCE_PATH = Path('../data/processed/yfinance')
WIKIPEDIA_PATH = Path('../data/processed/wikipedia')
FRED_PATH = Path('../data/processed/fred')
INTEGRATED_PATH = Path('../data/processed/integrated')

In [3]:
if not INTEGRATED_PATH.exists():
    INTEGRATED_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"{INTEGRATED_PATH} already exists.")

..\data\processed\integrated already exists.


# Load Cleaned Data

In [4]:
adj_close = pd.read_csv(
    YFINANCE_PATH / 'yfinance_adjusted_close_clean.csv',
    index_col=0,
    parse_dates=True
)

In [5]:
daily_returns = pd.read_csv(
    YFINANCE_PATH / 'yfinance_daily_returns_clean.csv',
    index_col=0,
    parse_dates=True
)

In [6]:
summary_stats = pd.read_csv(
    YFINANCE_PATH / "yfinance_processed_summary_stats.csv",
    index_col=0
)

In [7]:
risk_free_rate = pd.read_csv(
    FRED_PATH / "fred_risk_free_rate_processed.csv",
    index_col="date",
    parse_dates=True
)

In [8]:
cpi = pd.read_csv(
    FRED_PATH / "fred_cpi_processed.csv",
    index_col="date",
    parse_dates=True
)

In [9]:
sp500_metadata = pd.read_csv(
    WIKIPEDIA_PATH / "wikipedia_sp500_constituents_clean.csv"
)

# Inspect Data Shapes

In [10]:
print("Adjusted close:", adj_close.shape)
print("Daily returns:", daily_returns.shape)
print("Summary stats:", summary_stats.shape)
print("Risk-free rate:", risk_free_rate.shape)
print("CPI:", cpi.shape)
print("S&P 500 metadata:", sp500_metadata.shape)

Adjusted close: (2010, 21)
Daily returns: (2009, 21)
Summary stats: (21, 6)
Risk-free rate: (2010, 2)
CPI: (2892, 2)
S&P 500 metadata: (503, 3)


Now all cleaned datasets are loaded from their folders. The next step is to check whether the dates and ticker symbols are lined up correctly

# Check Date Alignment

In [11]:
print("Adjusted close date range:", adj_close.index.min(), "to", adj_close.index.max())
print("Daily returns date range:", daily_returns.index.min(), "to", daily_returns.index.max())
print("Risk-free rate date range:", risk_free_rate.index.min(), "to", risk_free_rate.index.max())
print("CPI date range:", cpi.index.min(), "to", cpi.index.max())

Adjusted close date range: 2018-01-02 00:00:00 to 2025-12-30 00:00:00
Daily returns date range: 2018-01-03 00:00:00 to 2025-12-30 00:00:00
Risk-free rate date range: 2018-01-02 00:00:00 to 2025-12-30 00:00:00
CPI date range: 2018-01-01 00:00:00 to 2025-12-01 00:00:00


In [12]:
print("Prices and risk-free rate same dates:", adj_close.index.equals(risk_free_rate.index))

# NOTE: Daily returns only need to be in the risk-free rate dates, not necessarily the same dates as prices
print("Daily returns dates are in risk-free rate:", daily_returns.index.isin(risk_free_rate.index).all())

Prices and risk-free rate same dates: True
Daily returns dates are in risk-free rate: True


The risk-free rate data is asligned to the same trading dates as the yfinance price data. Dialy returns start one day later because returns require a previous price to calculate the return. Therefore, the daily returns data will not have the same dates as the price data, but it should be a subset of the risk-free rate dates.

# Create Asset Metadata

In [13]:
portfolio_tickers = list(adj_close.columns)
print("Portfolio tickers:", portfolio_tickers)

Portfolio tickers: ['AAPL', 'AGG', 'AMZN', 'CAT', 'GLD', 'JPM', 'KO', 'LLY', 'LMT', 'MSFT', 'NEE', 'PG', 'QQQ', 'SPY', 'TLT', 'UNH', 'V', 'VNQ', 'VXUS', 'VZ', 'XOM']


In [14]:
asset_metadata = pd.DataFrame({
    "ticker": portfolio_tickers,
})

asset_metadata.head()

,ticker
0,AAPL
1,AGG
2,AMZN
3,CAT
4,GLD


# Merge Wikipedia Sector Metadata with Portfolio Tickers

In [15]:
asset_metadata = asset_metadata.merge(
    sp500_metadata,
    on="ticker",
    how="left",
)

In [16]:
asset_metadata

,ticker,company_name,gics_sector
0,AAPL,Apple Inc.,Information Technology
1,AGG,NaN,NaN
2,AMZN,Amazon,Consumer Discretionary
3,CAT,Caterpillar Inc.,Industrials
4,GLD,NaN,NaN
5,JPM,JPMorgan Chase,Financials
6,KO,Coca-Cola Company (The),Consumer Staples
7,LLY,Lilly (Eli),Health Care
8,LMT,Lockheed Martin,Industrials
9,MSFT,Microsoft,Information Technology


# 9 Manually Fill in ETF / Non-S&P Metadata

In [17]:
# NOTE: I'm filling in data with what's stated on the ETF's website. This is a manual process and may not be perfectly accurate.
manual_metadata = {
    "AGG": {
        "company_name": "iShares Core U.S. Aggregate Bond ETF",
        "gics_sector": "Bonds"
    },
    "TLT": {
        "company_name": "iShares 20+ Year Treasury Bond ETF",
        "gics_sector": "Long-Term Treasury Bonds"
    },
    "GLD": {
        "company_name": "SPDR Gold Shares",
        "gics_sector": "Gold / Commodities"
    },
    "QQQ": {
        "company_name": "Invesco QQQ Trust",
        "gics_sector": "Growth / Nasdaq ETF"
    },
    "SPY": {
        "company_name": "SPDR S&P 500 ETF Trust",
        "gics_sector": "Benchmark / S&P 500 ETF"
    },
    "VNQ": {
        "company_name": "Vanguard Real Estate ETF",
        "gics_sector": "Real Estate ETF"
    },
    "VXUS": {
        "company_name": "Vanguard Total International Stock ETF",
        "gics_sector": "International Equity ETF"
    }
}

In [18]:
for ticker, metadata in manual_metadata.items():
    mask = asset_metadata["ticker"] == ticker
    
    asset_metadata.loc[mask, "company_name"] = asset_metadata.loc[mask, "company_name"].fillna(metadata["company_name"])
    asset_metadata.loc[mask, "gics_sector"] = asset_metadata.loc[mask, "gics_sector"].fillna(metadata["gics_sector"])

In [19]:
asset_metadata

,ticker,company_name,gics_sector
0,AAPL,Apple Inc.,Information Technology
1,AGG,iShares Core U.S. Aggregate Bond ETF,Bonds
2,AMZN,Amazon,Consumer Discretionary
3,CAT,Caterpillar Inc.,Industrials
4,GLD,SPDR Gold Shares,Gold / Commodities
5,JPM,JPMorgan Chase,Financials
6,KO,Coca-Cola Company (The),Consumer Staples
7,LLY,Lilly (Eli),Health Care
8,LMT,Lockheed Martin,Industrials
9,MSFT,Microsoft,Information Technology


# Double Check for Missing Metadata

In [20]:
asset_metadata.isna().sum()

ticker          0
company_name    0
gics_sector     0
dtype: int64

In [21]:
asset_metadata[asset_metadata.isna().any(axis=1)]

,ticker,company_name,gics_sector


Since some of the portfolios were ETFS and didn't have data they didn't appear in the S&p 500 company table. So those assets were manually labeled by data founded on Google so every ticker has a name and category for analysis. Then we validated that all tickers have a name and category by checking for missing values. The final asset metadata table is now ready to be used for analysis. 

# Add Asset Type To Data

In [22]:
ETF_TICKERS = ["AGG", "TLT", "GLD", "QQQ", "SPY", "VNQ", "VXUS"]

In [23]:
asset_metadata["asset_type"] = np.where(
    asset_metadata["ticker"].isin(ETF_TICKERS),
    "ETF",
    "Stock"
)

asset_metadata

,ticker,company_name,gics_sector,asset_type
0,AAPL,Apple Inc.,Information Technology,Stock
1,AGG,iShares Core U.S. Aggregate Bond ETF,Bonds,ETF
2,AMZN,Amazon,Consumer Discretionary,Stock
3,CAT,Caterpillar Inc.,Industrials,Stock
4,GLD,SPDR Gold Shares,Gold / Commodities,ETF
5,JPM,JPMorgan Chase,Financials,Stock
6,KO,Coca-Cola Company (The),Consumer Staples,Stock
7,LLY,Lilly (Eli),Health Care,Stock
8,LMT,Lockheed Martin,Industrials,Stock
9,MSFT,Microsoft,Information Technology,Stock


# Create Long Price Dataset

In [24]:
prices_long = adj_close.reset_index().melt(
    id_vars="Date",
    var_name="ticker",
    value_name="adjusted_close"
)

In [25]:
prices_long.head()

,Date,ticker,adjusted_close
0,2018-01-02,AAPL,40.267082
1,2018-01-03,AAPL,40.260067
2,2018-01-04,AAPL,40.447075
3,2018-01-05,AAPL,40.907566
4,2018-01-08,AAPL,40.755634


# Create Long Returns Dataset

In [26]:
returns_long = daily_returns.reset_index().melt(
    id_vars="Date",
    var_name="ticker",
    value_name="daily_return"
)

In [27]:
returns_long.head()

,Date,ticker,daily_return
0,2018-01-03,AAPL,-0.000174
1,2018-01-04,AAPL,0.004645
2,2018-01-05,AAPL,0.011385
3,2018-01-08,AAPL,-0.003714
4,2018-01-09,AAPL,-0.000114


# Merge Price and Returns

In [28]:
daily_market_data = prices_long.merge(
    returns_long,
    on=["Date", "ticker"],
    how="left"
)

In [29]:
daily_market_data.head()

,Date,ticker,adjusted_close,daily_return
0,2018-01-02,AAPL,40.267082,NaN
1,2018-01-03,AAPL,40.260067,-0.000174
2,2018-01-04,AAPL,40.447075,0.004645
3,2018-01-05,AAPL,40.907566,0.011385
4,2018-01-08,AAPL,40.755634,-0.003714


The daily market data is converted into longer format. Which means each row now represents on one date. This will make it easier to merge with metadata, risk-free rate data, and future modeling featurs.

# Add Risk-Free Rate

In [30]:
risk_free_rate_reset = risk_free_rate.reset_index().rename(columns={"date": "Date", "risk_free_rate": "risk_free_rate"})

In [31]:
daily_market_data = daily_market_data.merge(
    risk_free_rate_reset,
    on="Date",
    how="left"
)

In [32]:
daily_market_data.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal
0,2018-01-02,AAPL,40.267082,NaN,1.44,0.0144
1,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141
2,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141
3,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139
4,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145


# Add Consumer Price Index (CPI) Data

In [33]:
# CPI is a monthly FRED series. Recalculate monthly inflation before joining to daily stock rows.
cpi_monthly = (
    cpi[["cpi_index"]]
    .resample("MS")
    .first()
    .assign(cpi_pct_change=lambda df: df["cpi_index"].pct_change().fillna(0))
    .reset_index()
    .rename(columns={"date": "cpi_month"})
)

In [34]:
daily_market_data["cpi_month"] = daily_market_data["Date"].dt.to_period("M").dt.to_timestamp()

daily_market_data = (
    daily_market_data
    .merge(cpi_monthly, on="cpi_month", how="left")
    .drop(columns="cpi_month")
)

In [35]:
daily_market_data.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change
0,2018-01-02,AAPL,40.267082,NaN,1.44,0.0144,248.859,0.0
1,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0
2,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0
3,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0
4,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0


# Add Asset Metadata

In [36]:
daily_market_data = daily_market_data.merge(
    asset_metadata,
    on="ticker",
    how="left"
)

In [37]:
daily_market_data.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type
0,2018-01-02,AAPL,40.267082,NaN,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock
1,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
2,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
3,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock
4,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock


Validate Final Shape

In [38]:
daily_market_data.shape

(42210, 11)

In [39]:
daily_market_data.isna().sum()

Date                       0
ticker                     0
adjusted_close             0
daily_return              21
risk_free_rate_pct         0
risk_free_rate_decimal     0
cpi_index                  0
cpi_pct_change             0
company_name               0
gics_sector                0
asset_type                 0
dtype: int64

In [40]:
daily_market_data["ticker"].nunique()

21

In [41]:
daily_market_data["ticker"].value_counts().head()

ticker
AAPL    2010
AGG     2010
AMZN    2010
CAT     2010
GLD     2010
Name: count, dtype: int64

This is now the integrated daily dataset that combines market price, returns, FRED macro/risk-free-rate data, and asset metadata. Missing daily returns on first day are expected because returns need a previous day.

# Create Modeling Baseline Dataset

In [42]:
modeling_base = daily_market_data.dropna(subset=["daily_return"]).copy()

In [43]:
modeling_base.shape

(42189, 11)

In [44]:
modeling_base.isna().sum()

Date                      0
ticker                    0
adjusted_close            0
daily_return              0
risk_free_rate_pct        0
risk_free_rate_decimal    0
cpi_index                 0
cpi_pct_change            0
company_name              0
gics_sector               0
asset_type                0
dtype: int64

The modeling base dataset removes rows without daily returns. This dataset is ready for feature engineering because each row has a date, ticker, price, return, risk-free rate, CPI context, and asset metadata.

# Save Integrated File

In [45]:
asset_metadata.to_csv(
    INTEGRATED_PATH / "asset_metadata.csv",
    index=False
)

In [46]:
daily_market_data.to_csv(
    INTEGRATED_PATH / "daily_market_data.csv",
    index=False
)

In [47]:
modeling_base.to_csv(
    INTEGRATED_PATH / "modeling_base_dataset.csv",
    index=False
)

In [48]:
print("Saved files:")
print(INTEGRATED_PATH / "asset_metadata.csv")
print(INTEGRATED_PATH / "daily_market_data.csv")
print(INTEGRATED_PATH / "modeling_base_dataset.csv")

Saved files:
..\data\processed\integrated\asset_metadata.csv
..\data\processed\integrated\daily_market_data.csv
..\data\processed\integrated\modeling_base_dataset.csv


# Final Check

In [49]:
asset_metadata.head()

,ticker,company_name,gics_sector,asset_type
0,AAPL,Apple Inc.,Information Technology,Stock
1,AGG,iShares Core U.S. Aggregate Bond ETF,Bonds,ETF
2,AMZN,Amazon,Consumer Discretionary,Stock
3,CAT,Caterpillar Inc.,Industrials,Stock
4,GLD,SPDR Gold Shares,Gold / Commodities,ETF


In [50]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type
1,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
2,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
3,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock
4,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock
5,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock


# Final Summary

## Data Integration

In this notebook, we combined the cleaned outputs from the three project data sources:

- Yahoo Finance price and return data
- Wikipedia S&P 500 company metadata
- FRED risk-free rate and CPI macroeconomic data

The final integrated files are:

- `data/processed/integrated/asset_metadata.csv`
- `data/processed/integrated/daily_market_data.csv`
- `data/processed/integrated/modeling_base_dataset.csv`

These datasets will be used in the next step for feature engineering and predictive modeling.